# ODI to Databricks Migration: WC_BADGE_DETAILS_D

**Source File:** `WC_BADGE_DETAILS_D.txt`
**Conversion Timestamp:** 2023-10-27T10:00:00Z

This notebook contains the Spark SQL conversion of an ODI session for loading the `WC_BADGE_DETAILS_D` dimension table. It handles staging, flow, and merge operations, including specific ODI patterns like MAX-based deduplication and `IND_UPDATE` flagging.

In [ ]:
# COMMAND ----------
dbutils.widgets.text("V_ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("V_ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00.000000", "Last Extract Time (YYYY-MM-DD HH:MM:SS.ffffff)")
dbutils.widgets.text("V_ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59.999999", "Current Extract Time (YYYY-MM-DD HH:MM:SS.ffffff)")
dbutils.widgets.text("V_ROW_WID", "-1", "Row WID Parameter")
dbutils.widgets.text("DATASOURCE_NUM_ID", "380", "Datasource Number ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

The following temporary views are created to hold ETL parameter values, dynamically fetched or provided via widgets.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {2}: Get ETL last extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT
    to_timestamp(COALESCE(
        MAX(etl_last_extract_time),
        to_timestamp('${V_ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss.SSSSSS')
    ), 'yyyy-MM-dd HH:mm:ss.SSSSSS') AS etl_last_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE etl_job_type = '${V_ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {3}: Get ETL current extract time
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT
    to_timestamp(COALESCE(
        MAX(etl_current_extract_time),
        to_timestamp('${V_ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss.SSSSSS')
    ), 'yyyy-MM-dd HH:mm:ss.SSSSSS') AS etl_current_extract_time
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE etl_job_type = '${V_ETL_JOB_TYPE}';

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {6}: Get ETL ROW_WID parameter
CREATE OR REPLACE TEMPORARY VIEW v_etl_row_wid_param AS
SELECT COALESCE(MAX(CAST(etl_row_wid_param AS BIGINT)), CAST('${V_ROW_WID}' AS BIGINT)) AS etl_row_wid_param
FROM workspace.prxbi_dw.wc_etl_parameters
WHERE etl_job_type = '${V_ETL_JOB_TYPE}';

In [ ]:
# COMMAND ----------
print("ETL Parameters:")
display(spark.sql("SELECT '${V_ETL_JOB_TYPE}' AS V_ETL_JOB_TYPE,
                          (SELECT etl_last_extract_time
FROM v_etl_last_extract_time) AS V_ETL_LAST_EXTRACT_TIME,
                          (SELECT etl_current_extract_time
FROM v_etl_current_extract_time) AS V_ETL_CURRENT_EXTRACT_TIME,
                          (SELECT etl_row_wid_param
FROM v_etl_row_wid_param) AS V_ROW_WID,
                          '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
                          '${ODI_SESS_NO}' AS ODI_SESS_NO"))

## Staging Table (`c_badge_ts_stg`)

This section handles the creation, loading, and optimization of the staging table (`C$_...`).

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {30}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_ts_stg;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.prxbi_dw.c_badge_ts_stg (
	ID	STRING,
	BADGELOCATION	STRING,
	BADGETOKEN	STRING,
	BADGEVERSION	BIGINT,
	CONTACTEMAIL	STRING,
	CONTACTFIRSTNAME	STRING,
	CONTACTJOBTITLE	STRING,
	CONTACTLASTNAME	STRING,
	CONTACTPERSONRXMASTERID	STRING,
	CREATEDBYREGISTRATIONTYPE	STRING,
	CREATEDBYTYPE	STRING,
	CULTURE	STRING,
	CUSTOMERTYPE	STRING,
	EVENTEDITIONGBSCODE	STRING,
	ISBADGEUPDATE	STRING,
	MARKETINGPREFERENCESPROMPTREQU	STRING,
	ORGANISATIONCITY	STRING,
	ORGANISATIONCOUNTRYCODE	STRING,
	ORGANISATIONDISPLAYNAME	STRING,
	ORGANISATIONRXMASTERID	STRING,
	ORGANISATIONSTATE	STRING,
	PARTICIPATINGORGANISATIONID	STRING,
	PRODUCTCODE	STRING,
	QRCODECONTENT	STRING,
	REGISTRATIONID	STRING,
	STATUS	BIGINT,
	SUPPORTSTAFFCOMPANYADDRESS	STRING,
	SUPPORTSTAFFCOMPANYNAME	STRING,
	SUPPORTSTAFFMOBILEPHONE	STRING,
	SUPPORTSTAFFREPORTSTO	STRING,
	SUPPORTSTAFFSTANDS	STRING,
	SUPPORTSTAFFUSERACCESS	STRING,
	VERSIONNUMBER	BIGINT,
	MOBILEPHONE	STRING,
	FIRSTSCANNEDDATE	TIMESTAMP,
	LASTPRINTEDDATE	TIMESTAMP,
	ACCESSVALIDITYMODIFIEDDATE	TIMESTAMP,
	CREATEDDATE	TIMESTAMP,
	COMPANYPRODUCTCODE	STRING,
	PAYMENTSTATUS	STRING,
	PHOTOKEY	STRING,
	PHOTOSOURCE	STRING,
	PHOTOSOURCETYPE	STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {50}: Insert into staging
-- Rewritten ODI MAX self-join dedup using ROW_NUMBER() as per F.12
INSERT INTO workspace.prxbi_dw.c_badge_ts_stg (
	ID,
	BADGELOCATION,
	BADGETOKEN,
	BADGEVERSION,
	CONTACTEMAIL,
	CONTACTFIRSTNAME,
	CONTACTJOBTITLE,
	CONTACTLASTNAME,
	CONTACTPERSONRXMASTERID,
	CREATEDBYREGISTRATIONTYPE,
	CREATEDBYTYPE,
	CULTURE,
	CUSTOMERTYPE,
	EVENTEDITIONGBSCODE,
	ISBADGEUPDATE,
	MARKETINGPREFERENCESPROMPTREQU,
	ORGANISATIONCITY,
	ORGANISATIONCOUNTRYCODE,
	ORGANISATIONDISPLAYNAME,
	ORGANISATIONRXMASTERID,
	ORGANISATIONSTATE,
	PARTICIPATINGORGANISATIONID,
	PRODUCTCODE,
	QRCODECONTENT,
	REGISTRATIONID,
	STATUS,
	SUPPORTSTAFFCOMPANYADDRESS,
	SUPPORTSTAFFCOMPANYNAME,
	SUPPORTSTAFFMOBILEPHONE,
	SUPPORTSTAFFREPORTSTO,
	SUPPORTSTAFFSTANDS,
	SUPPORTSTAFFUSERACCESS,
	VERSIONNUMBER,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE
)
SELECT
	ID,
	BADGELOCATION,
	BADGETOKEN,
	BADGEVERSION,
	CONTACTEMAIL,
	CONTACTFIRSTNAME,
	CONTACTJOBTITLE,
	CONTACTLASTNAME,
	CONTACTPERSONRXMASTERID,
	CREATEDBYREGISTRATIONTYPE,
	CREATEDBYTYPE,
	CULTURE,
	CUSTOMERTYPE,
	EVENTEDITIONGBSCODE,
	ISBADGEUPDATE,
	MARKETINGPREFERENCESPROMPTREQU,
	ORGANISATIONCITY,
	ORGANISATIONCOUNTRYCODE,
	ORGANISATIONDISPLAYNAME,
	ORGANISATIONRXMASTERID,
	ORGANISATIONSTATE,
	PARTICIPATINGORGANISATIONID,
	PRODUCTCODE,
	QRCODECONTENT,
	REGISTRATIONID,
	STATUS,
	SUPPORTSTAFFCOMPANYADDRESS,
	SUPPORTSTAFFCOMPANYNAME,
	SUPPORTSTAFFMOBILEPHONE,
	SUPPORTSTAFFREPORTSTO,
	SUPPORTSTAFFSTANDS,
	SUPPORTSTAFFUSERACCESS,
	VERSIONNUMBER,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE
FROM (
    SELECT
        AMERCURY_BADGE_TS.*,
        ROW_NUMBER() OVER (
            PARTITION BY AMERCURY_BADGE_TS.ID
            ORDER BY AMERCURY_BADGE_TS.INT_INSERT_DATE DESC, AMERCURY_BADGE_TS.VERSIONNUMBER DESC
        ) AS rn
    FROM workspace.prxbi_ts.wc_mercury_badge_ts AMERCURY_BADGE_TS
    WHERE
        AMERCURY_BADGE_TS.INT_INSERT_DATE > (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
        AND AMERCURY_BADGE_TS.INT_INSERT_DATE <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
) AS filtered_source
WHERE rn = 1;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {60}: Gather table statistics ->
OPTIMIZE -- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.c_badge_ts_stg
ZORDER BY (ID, VERSIONNUMBER);

## Flow Table (`i_wc_badge_details_d_flow`)

This section prepares the flow table, performing transformations and flagging records for update or insert.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {80}: Drop flow table (removed, this is a marker task)
-- SCEN_TASK_NO {90}: Create flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;
CREATE TABLE workspace.prxbi_dw.i_wc_badge_details_d_flow (
	ROW_WID		BIGINT,
	BADGE_ID		STRING,
	BADGE_LOCATION		STRING,
	BADGE_TOKEN		STRING,
	BADGE_VERSION		BIGINT,
	CONTACT_EMAIL		STRING,
	CONTACT_FIRST_NAME		STRING,
	CONTACT_LAST_NAME		STRING,
	CONTACT_JOB_TITLE		STRING,
	CONTACT_PERSON_ID		STRING,
	CREATION_REG_TYPE		STRING,
	CREATION_TYPE		STRING,
	CULTURE		STRING,
	CUSTOMER_TYPE		STRING,
	EVENT_EDITION_CODE		STRING,
	BADGE_UPDATE_FLG		STRING,
	MARKETING_PREF_PROMPT		STRING,
	ORG_NAME		STRING,
	ORG_CITY		STRING,
	ORG_COUNTRY		STRING,
	ORG_ID		STRING,
	ORG_STATE		STRING,
	PARTICIPATING_ORG_ID		STRING,
	PRODUCT_CODE		STRING,
	QR_CODE		STRING,
	REGISTRATION_ID		STRING,
	STATUS		BIGINT,
	STAFF_COMPANY_NAME		STRING,
	STAFF_COMPANY_ADDR		STRING,
	STAFF_PHONE_NUM		STRING,
	STAFF_REPORTING		STRING,
	STAFF_STANDS		STRING,
	STAFF_USER_ACCESS		STRING,
	VERSION_NUM		BIGINT,
	INTEGRATION_ID		STRING,
	DATASOURCE_NUM_ID		STRING,
	W_INSERT_DT		TIMESTAMP,
	W_UPDATE_DT		TIMESTAMP,
	MOBILEPHONE		STRING,
	FIRSTSCANNEDDATE		TIMESTAMP,
	LASTPRINTEDDATE		TIMESTAMP,
	FIRSTSCANNEDDATE_FLG		STRING,
	LASTPRINTEDDATE_FLG		STRING,
	ACCESSVALIDITYMODIFIEDDATE		TIMESTAMP,
	CREATEDDATE		TIMESTAMP,
	COMPANYPRODUCTCODE		STRING,
	PAYMENTSTATUS		STRING,
	PHOTOKEY		STRING,
	PHOTOSOURCE		STRING,
	PHOTOSOURCETYPE		STRING,
	PACKAGE_NAME		STRING,
	IND_UPDATE		STRING
) USING DELTA;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {100}: Insert into flow table with initial IND_UPDATE = 'I'
-- The original `NOT EXISTS` clause with full column comparison is preserved using `IS NOT DISTINCT FROM`.
INSERT INTO workspace.prxbi_dw.i_wc_badge_details_d_flow (
	BADGE_ID,
	BADGE_LOCATION,
	BADGE_TOKEN,
	BADGE_VERSION,
	CONTACT_EMAIL,
	CONTACT_FIRST_NAME,
	CONTACT_LAST_NAME,
	CONTACT_JOB_TITLE,
	CONTACT_PERSON_ID,
	CREATION_REG_TYPE,
	CREATION_TYPE,
	CULTURE,
	CUSTOMER_TYPE,
	EVENT_EDITION_CODE,
	BADGE_UPDATE_FLG,
	MARKETING_PREF_PROMPT,
	ORG_NAME,
	ORG_CITY,
	ORG_COUNTRY,
	ORG_ID,
	ORG_STATE,
	PARTICIPATING_ORG_ID,
	PRODUCT_CODE,
	QR_CODE,
	REGISTRATION_ID,
	STATUS,
	STAFF_COMPANY_NAME,
	STAFF_COMPANY_ADDR,
	STAFF_PHONE_NUM,
	STAFF_REPORTING,
	STAFF_STANDS,
	STAFF_USER_ACCESS,
	VERSION_NUM,
	INTEGRATION_ID,
	DATASOURCE_NUM_ID,
	MOBILEPHONE,
	FIRSTSCANNEDDATE,
	LASTPRINTEDDATE,
	FIRSTSCANNEDDATE_FLG,
	LASTPRINTEDDATE_FLG,
	ACCESSVALIDITYMODIFIEDDATE,
	CREATEDDATE,
	COMPANYPRODUCTCODE,
	PAYMENTSTATUS,
	PHOTOKEY,
	PHOTOSOURCE,
	PHOTOSOURCETYPE,
	PACKAGE_NAME,
	IND_UPDATE,
	W_INSERT_DT,
	W_UPDATE_DT
)
SELECT
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    'I' AS IND_UPDATE,
    current_timestamp() AS W_INSERT_DT,
    current_timestamp() AS W_UPDATE_DT
FROM (
    SELECT
        JOIN1_A.ID AS BADGE_ID,
        JOIN1_A.BADGELOCATION AS BADGE_LOCATION,
        JOIN1_A.BADGETOKEN AS BADGE_TOKEN,
        JOIN1_A.BADGEVERSION AS BADGE_VERSION,
        JOIN1_A.CONTACTEMAIL AS CONTACT_EMAIL,
        JOIN1_A.CONTACTFIRSTNAME AS CONTACT_FIRST_NAME,
        JOIN1_A.CONTACTLASTNAME AS CONTACT_LAST_NAME,
        JOIN1_A.CONTACTJOBTITLE AS CONTACT_JOB_TITLE,
        JOIN1_A.CONTACTPERSONRXMASTERID AS CONTACT_PERSON_ID,
        JOIN1_A.CREATEDBYREGISTRATIONTYPE AS CREATION_REG_TYPE,
        JOIN1_A.CREATEDBYTYPE AS CREATION_TYPE,
        JOIN1_A.CULTURE AS CULTURE,
        JOIN1_A.CUSTOMERTYPE AS CUSTOMER_TYPE,
        JOIN1_A.EVENTEDITIONGBSCODE AS EVENT_EDITION_CODE,
        JOIN1_A.ISBADGEUPDATE AS BADGE_UPDATE_FLG,
        JOIN1_A.MARKETINGPREFERENCESPROMPTREQU AS MARKETING_PREF_PROMPT,
        JOIN1_A.ORGANISATIONDISPLAYNAME AS ORG_NAME,
        JOIN1_A.ORGANISATIONCITY AS ORG_CITY,
        JOIN1_A.ORGANISATIONCOUNTRYCODE AS ORG_COUNTRY,
        JOIN1_A.ORGANISATIONRXMASTERID AS ORG_ID,
        JOIN1_A.ORGANISATIONSTATE AS ORG_STATE,
        JOIN1_A.PARTICIPATINGORGANISATIONID AS PARTICIPATING_ORG_ID,
        JOIN1_A.PRODUCTCODE AS PRODUCT_CODE,
        JOIN1_A.QRCODECONTENT AS QR_CODE,
        JOIN1_A.REGISTRATIONID AS REGISTRATION_ID,
        JOIN1_A.STATUS AS STATUS,
        JOIN1_A.SUPPORTSTAFFCOMPANYNAME AS STAFF_COMPANY_NAME,
        JOIN1_A.SUPPORTSTAFFCOMPANYADDRESS AS STAFF_COMPANY_ADDR,
        JOIN1_A.SUPPORTSTAFFMOBILEPHONE AS STAFF_PHONE_NUM,
        JOIN1_A.SUPPORTSTAFFREPORTSTO AS STAFF_REPORTING,
        JOIN1_A.SUPPORTSTAFFSTANDS AS STAFF_STANDS,
        JOIN1_A.SUPPORTSTAFFUSERACCESS AS STAFF_USER_ACCESS,
        JOIN1_A.VERSIONNUMBER AS VERSION_NUM,
        JOIN1_A.ID AS INTEGRATION_ID,
        '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,
        JOIN1_A.MOBILEPHONE AS MOBILEPHONE,
        JOIN1_A.FIRSTSCANNEDDATE AS FIRSTSCANNEDDATE,
        JOIN1_A.LASTPRINTEDDATE AS LASTPRINTEDDATE,
        CASE WHEN JOIN1_A.FIRSTSCANNEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS FIRSTSCANNEDDATE_FLG,
        CASE WHEN JOIN1_A.LASTPRINTEDDATE IS NOT NULL THEN 'Y' ELSE 'N' END AS LASTPRINTEDDATE_FLG,
        JOIN1_A.ACCESSVALIDITYMODIFIEDDATE AS ACCESSVALIDITYMODIFIEDDATE,
        JOIN1_A.CREATEDDATE AS CREATEDDATE,
        JOIN1_A.COMPANYPRODUCTCODE AS COMPANYPRODUCTCODE,
        JOIN1_A.PAYMENTSTATUS AS PAYMENTSTATUS,
        JOIN1_A.PHOTOKEY AS PHOTOKEY,
        JOIN1_A.PHOTOSOURCE AS PHOTOSOURCE,
        JOIN1_A.PHOTOSOURCETYPE AS PHOTOSOURCETYPE,
        WC_BADGE_PRODUCT_D_2.NAME_1 AS PACKAGE_NAME
    FROM workspace.prxbi_dw.c_badge_ts_stg AS JOIN1_A
    LEFT OUTER JOIN (
        SELECT
            WC_BADGE_PRODUCT_D_1.ID AS ID,
            WC_BADGE_PRODUCT_D_1.SKU AS SKU,
            WC_BADGE_PRODUCT_D_1.NAME AS NAME,
            WC_BADGE_PRODUCT_D_1.SKU_1 AS SKU_1,
            WC_BADGE_PRODUCT_D_1.NAME_1 AS NAME_1
        FROM (
            SELECT
                WC_BADGE_PRODUCT_D.ID AS ID,
                WC_BADGE_PRODUCT_D.SKU AS SKU,
                WC_BADGE_PRODUCT_D.NAME AS NAME,
                RANK() OVER (PARTITION BY WC_BADGE_PRODUCT_D.SKU ORDER BY WC_BADGE_PRODUCT_D.ID DESC) AS COL,
                WC_BADGE_PRODUCT_D.SKU AS SKU_1,
                WC_BADGE_PRODUCT_D.NAME AS NAME_1
            FROM workspace.prxbi_dw.wc_badge_product_d AS WC_BADGE_PRODUCT_D
        ) AS WC_BADGE_PRODUCT_D_1
        WHERE WC_BADGE_PRODUCT_D_1.COL = 1
    ) AS WC_BADGE_PRODUCT_D_2
    ON JOIN1_A.PRODUCTCODE = WC_BADGE_PRODUCT_D_2.SKU_1
    WHERE (1=1)
) AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.prxbi_dw.wc_badge_details_d AS T
    WHERE
        T.INTEGRATION_ID = S.INTEGRATION_ID
        AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
        AND T.BADGE_ID IS NOT DISTINCT FROM S.BADGE_ID
        AND T.BADGE_LOCATION IS NOT DISTINCT FROM S.BADGE_LOCATION
        AND T.BADGE_TOKEN IS NOT DISTINCT FROM S.BADGE_TOKEN
        AND T.BADGE_VERSION IS NOT DISTINCT FROM S.BADGE_VERSION
        AND T.CONTACT_EMAIL IS NOT DISTINCT FROM S.CONTACT_EMAIL
        AND T.CONTACT_FIRST_NAME IS NOT DISTINCT FROM S.CONTACT_FIRST_NAME
        AND T.CONTACT_LAST_NAME IS NOT DISTINCT FROM S.CONTACT_LAST_NAME
        AND T.CONTACT_JOB_TITLE IS NOT DISTINCT FROM S.CONTACT_JOB_TITLE
        AND T.CONTACT_PERSON_ID IS NOT DISTINCT FROM S.CONTACT_PERSON_ID
        AND T.CREATION_REG_TYPE IS NOT DISTINCT FROM S.CREATION_REG_TYPE
        AND T.CREATION_TYPE IS NOT DISTINCT FROM S.CREATION_TYPE
        AND T.CULTURE IS NOT DISTINCT FROM S.CULTURE
        AND T.CUSTOMER_TYPE IS NOT DISTINCT FROM S.CUSTOMER_TYPE
        AND T.EVENT_EDITION_CODE IS NOT DISTINCT FROM S.EVENT_EDITION_CODE
        AND T.BADGE_UPDATE_FLG IS NOT DISTINCT FROM S.BADGE_UPDATE_FLG
        AND T.MARKETING_PREF_PROMPT IS NOT DISTINCT FROM S.MARKETING_PREF_PROMPT
        AND T.ORG_NAME IS NOT DISTINCT FROM S.ORG_NAME
        AND T.ORG_CITY IS NOT DISTINCT FROM S.ORG_CITY
        AND T.ORG_COUNTRY IS NOT DISTINCT FROM S.ORG_COUNTRY
        AND T.ORG_ID IS NOT DISTINCT FROM S.ORG_ID
        AND T.ORG_STATE IS NOT DISTINCT FROM S.ORG_STATE
        AND T.PARTICIPATING_ORG_ID IS NOT DISTINCT FROM S.PARTICIPATING_ORG_ID
        AND T.PRODUCT_CODE IS NOT DISTINCT FROM S.PRODUCT_CODE
        AND T.QR_CODE IS NOT DISTINCT FROM S.QR_CODE
        AND T.REGISTRATION_ID IS NOT DISTINCT FROM S.REGISTRATION_ID
        AND T.STATUS IS NOT DISTINCT FROM S.STATUS
        AND T.STAFF_COMPANY_NAME IS NOT DISTINCT FROM S.STAFF_COMPANY_NAME
        AND T.STAFF_COMPANY_ADDR IS NOT DISTINCT FROM S.STAFF_COMPANY_ADDR
        AND T.STAFF_PHONE_NUM IS NOT DISTINCT FROM S.STAFF_PHONE_NUM
        AND T.STAFF_REPORTING IS NOT DISTINCT FROM S.STAFF_REPORTING
        AND T.STAFF_STANDS IS NOT DISTINCT FROM S.STAFF_STANDS
        AND T.STAFF_USER_ACCESS IS NOT DISTINCT FROM S.STAFF_USER_ACCESS
        AND T.VERSION_NUM IS NOT DISTINCT FROM S.VERSION_NUM
        AND T.MOBILEPHONE IS NOT DISTINCT FROM S.MOBILEPHONE
        AND T.FIRSTSCANNEDDATE IS NOT DISTINCT FROM S.FIRSTSCANNEDDATE
        AND T.LASTPRINTEDDATE IS NOT DISTINCT FROM S.LASTPRINTEDDATE
        AND T.FIRSTSCANNEDDATE_FLG IS NOT DISTINCT FROM S.FIRSTSCANNEDDATE_FLG
        AND T.LASTPRINTEDDATE_FLG IS NOT DISTINCT FROM S.LASTPRINTEDDATE_FLG
        AND T.ACCESSVALIDITYMODIFIEDDATE IS NOT DISTINCT FROM S.ACCESSVALIDITYMODIFIEDDATE
        AND T.CREATEDDATE IS NOT DISTINCT FROM S.CREATEDDATE
        AND T.COMPANYPRODUCTCODE IS NOT DISTINCT FROM S.COMPANYPRODUCTCODE
        AND T.PAYMENTSTATUS IS NOT DISTINCT FROM S.PAYMENTSTATUS
        AND T.PHOTOKEY IS NOT DISTINCT FROM S.PHOTOKEY
        AND T.PHOTOSOURCE IS NOT DISTINCT FROM S.PHOTOSOURCE
        AND T.PHOTOSOURCETYPE IS NOT DISTINCT FROM S.PHOTOSOURCETYPE
        AND T.PACKAGE_NAME IS NOT DISTINCT FROM S.PACKAGE_NAME
);

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {110}:
Create index ->
OPTIMIZE ZORDER BY
-- SCEN_TASK_NO {120}: Gather table statistics ->
OPTIMIZE -- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.i_wc_badge_details_d_flow
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## Mark Records for Update

This step identifies records in the flow table that correspond to existing records in the target dimension table and flags them for update (`IND_UPDATE = 'U'`).

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {130}: Flag records in flow table for update
-- Rewritten from UPDATE WHERE (col1, col2) IN (SELECT ...) to MERGE as per F.10
MERGE INTO workspace.prxbi_dw.i_wc_badge_details_d_flow AS T
USING (
    SELECT INTEGRATION_ID, DATASOURCE_NUM_ID
    FROM workspace.prxbi_dw.wc_badge_details_d
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

## MERGE into Target (`wc_badge_details_d`)

This section performs the final MERGE operation into the target dimension table, handling both updates and inserts based on the `IND_UPDATE` flag.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {150} (UPDATE) and {160} (INSERT) combined into a single MERGE statement
-- ROW_WID is excluded from INSERT list as it should be GENERATED ALWAYS AS IDENTITY.
MERGE INTO workspace.prxbi_dw.wc_badge_details_d AS T
USING workspace.prxbi_dw.i_wc_badge_details_d_flow AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.BADGE_ID = S.BADGE_ID,
    T.BADGE_LOCATION = S.BADGE_LOCATION,
    T.BADGE_TOKEN = S.BADGE_TOKEN,
    T.BADGE_VERSION = S.BADGE_VERSION,
    T.CONTACT_EMAIL = S.CONTACT_EMAIL,
    T.CONTACT_FIRST_NAME = S.CONTACT_FIRST_NAME,
    T.CONTACT_LAST_NAME = S.CONTACT_LAST_NAME,
    T.CONTACT_JOB_TITLE = S.CONTACT_JOB_TITLE,
    T.CONTACT_PERSON_ID = S.CONTACT_PERSON_ID,
    T.CREATION_REG_TYPE = S.CREATION_REG_TYPE,
    T.CREATION_TYPE = S.CREATION_TYPE,
    T.CULTURE = S.CULTURE,
    T.CUSTOMER_TYPE = S.CUSTOMER_TYPE,
    T.EVENT_EDITION_CODE = S.EVENT_EDITION_CODE,
    T.BADGE_UPDATE_FLG = S.BADGE_UPDATE_FLG,
    T.MARKETING_PREF_PROMPT = S.MARKETING_PREF_PROMPT,
    T.ORG_NAME = S.ORG_NAME,
    T.ORG_CITY = S.ORG_CITY,
    T.ORG_COUNTRY = S.ORG_COUNTRY,
    T.ORG_ID = S.ORG_ID,
    T.ORG_STATE = S.ORG_STATE,
    T.PARTICIPATING_ORG_ID = S.PARTICIPATING_ORG_ID,
    T.PRODUCT_CODE = S.PRODUCT_CODE,
    T.QR_CODE = S.QR_CODE,
    T.REGISTRATION_ID = S.REGISTRATION_ID,
    T.STATUS = S.STATUS,
    T.STAFF_COMPANY_NAME = S.STAFF_COMPANY_NAME,
    T.STAFF_COMPANY_ADDR = S.STAFF_COMPANY_ADDR,
    T.STAFF_PHONE_NUM = S.STAFF_PHONE_NUM,
    T.STAFF_REPORTING = S.STAFF_REPORTING,
    T.STAFF_STANDS = S.STAFF_STANDS,
    T.STAFF_USER_ACCESS = S.STAFF_USER_ACCESS,
    T.VERSION_NUM = S.VERSION_NUM,
    T.MOBILEPHONE = S.MOBILEPHONE,
    T.FIRSTSCANNEDDATE = S.FIRSTSCANNEDDATE,
    T.LASTPRINTEDDATE = S.LASTPRINTEDDATE,
    T.FIRSTSCANNEDDATE_FLG = S.FIRSTSCANNEDDATE_FLG,
    T.LASTPRINTEDDATE_FLG = S.LASTPRINTEDDATE_FLG,
    T.ACCESSVALIDITYMODIFIEDDATE = S.ACCESSVALIDITYMODIFIEDDATE,
    T.CREATEDDATE = S.CREATEDDATE,
    T.COMPANYPRODUCTCODE = S.COMPANYPRODUCTCODE,
    T.PAYMENTSTATUS = S.PAYMENTSTATUS,
    T.PHOTOKEY = S.PHOTOKEY,
    T.PHOTOSOURCE = S.PHOTOSOURCE,
    T.PHOTOSOURCETYPE = S.PHOTOSOURCETYPE,
    T.PACKAGE_NAME = S.PACKAGE_NAME,
    T.W_UPDATE_DT = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
    BADGE_ID,
    BADGE_LOCATION,
    BADGE_TOKEN,
    BADGE_VERSION,
    CONTACT_EMAIL,
    CONTACT_FIRST_NAME,
    CONTACT_LAST_NAME,
    CONTACT_JOB_TITLE,
    CONTACT_PERSON_ID,
    CREATION_REG_TYPE,
    CREATION_TYPE,
    CULTURE,
    CUSTOMER_TYPE,
    EVENT_EDITION_CODE,
    BADGE_UPDATE_FLG,
    MARKETING_PREF_PROMPT,
    ORG_NAME,
    ORG_CITY,
    ORG_COUNTRY,
    ORG_ID,
    ORG_STATE,
    PARTICIPATING_ORG_ID,
    PRODUCT_CODE,
    QR_CODE,
    REGISTRATION_ID,
    STATUS,
    STAFF_COMPANY_NAME,
    STAFF_COMPANY_ADDR,
    STAFF_PHONE_NUM,
    STAFF_REPORTING,
    STAFF_STANDS,
    STAFF_USER_ACCESS,
    VERSION_NUM,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    MOBILEPHONE,
    FIRSTSCANNEDDATE,
    LASTPRINTEDDATE,
    FIRSTSCANNEDDATE_FLG,
    LASTPRINTEDDATE_FLG,
    ACCESSVALIDITYMODIFIEDDATE,
    CREATEDDATE,
    COMPANYPRODUCTCODE,
    PAYMENTSTATUS,
    PHOTOKEY,
    PHOTOSOURCE,
    PHOTOSOURCETYPE,
    PACKAGE_NAME,
    W_INSERT_DT,
    W_UPDATE_DT
) VALUES (
    S.BADGE_ID,
    S.BADGE_LOCATION,
    S.BADGE_TOKEN,
    S.BADGE_VERSION,
    S.CONTACT_EMAIL,
    S.CONTACT_FIRST_NAME,
    S.CONTACT_LAST_NAME,
    S.CONTACT_JOB_TITLE,
    S.CONTACT_PERSON_ID,
    S.CREATION_REG_TYPE,
    S.CREATION_TYPE,
    S.CULTURE,
    S.CUSTOMER_TYPE,
    S.EVENT_EDITION_CODE,
    S.BADGE_UPDATE_FLG,
    S.MARKETING_PREF_PROMPT,
    S.ORG_NAME,
    S.ORG_CITY,
    S.ORG_COUNTRY,
    S.ORG_ID,
    S.ORG_STATE,
    S.PARTICIPATING_ORG_ID,
    S.PRODUCT_CODE,
    S.QR_CODE,
    S.REGISTRATION_ID,
    S.STATUS,
    S.STAFF_COMPANY_NAME,
    S.STAFF_COMPANY_ADDR,
    S.STAFF_PHONE_NUM,
    S.STAFF_REPORTING,
    S.STAFF_STANDS,
    S.STAFF_USER_ACCESS,
    S.VERSION_NUM,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.MOBILEPHONE,
    S.FIRSTSCANNEDDATE,
    S.LASTPRINTEDDATE,
    S.FIRSTSCANNEDDATE_FLG,
    S.LASTPRINTEDDATE_FLG,
    S.ACCESSVALIDITYMODIFIEDDATE,
    S.CREATEDDATE,
    S.COMPANYPRODUCTCODE,
    S.PAYMENTSTATUS,
    S.PHOTOKEY,
    S.PHOTOSOURCE,
    S.PHOTOSOURCETYPE,
    S.PACKAGE_NAME,
    current_timestamp(),
    current_timestamp()
);

## Optimize Target

Optimize the target table after the MERGE operation to improve query performance.

In [ ]:
-- MAGIC %sql
-- Disable
ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.prxbi_dw.wc_badge_details_d
ZORDER BY (INTEGRATION_ID, DATASOURCE_NUM_ID);

## Cleanup

Drop temporary staging and flow tables.

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {180}:
Drop flow table
DROP TABLE IF EXISTS workspace.prxbi_dw.i_wc_badge_details_d_flow;

In [ ]:
-- MAGIC %sql
-- SCEN_TASK_NO {210}:
Drop staging table
DROP TABLE IF EXISTS workspace.prxbi_dw.c_badge_ts_stg;

## Validation

Perform final checks and counts to validate the ETL process.

In [ ]:
-- MAGIC %sql
SELECT COUNT(*) AS final_record_count
FROM workspace.prxbi_dw.wc_badge_details_d
WHERE w_update_dt >= (SELECT etl_last_extract_time
FROM v_etl_last_extract_time)
AND w_update_dt <= (SELECT etl_current_extract_time
FROM v_etl_current_extract_time);

In [ ]:
-- MAGIC %sql
SELECT *
FROM workspace.prxbi_dw.wc_badge_details_d
WHERE w_update_dt >= (SELECT etl_last_extract_time FROM v_etl_last_extract_time)
  AND w_update_dt <= (SELECT etl_current_extract_time FROM v_etl_current_extract_time)
ORDER BY w_update_dt DESC
LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **`ROW_WID` Identity Column:** The `ROW_WID` column in `workspace.prxbi_dw.wc_badge_details_d` is assumed to be `GENERATED ALWAYS AS IDENTITY`. It has been removed from all `INSERT` statements as per Databricks best practices for identity columns.
2.  **`wc_etl_parameters` Table:** This notebook assumes the existence of a `workspace.prxbi_dw.wc_etl_parameters` table with `etl_last_extract_time`, `etl_current_extract_time`, `etl_row_wid_param`, and `etl_job_type` columns to manage ETL state. This table needs to be set up and maintained externally if it doesn't already exist.
3.  **Error Handling (E$ Tables):** The original ODI script did not include `E$_` table operations in the provided tasks. If error logging is required, corresponding `CREATE TABLE IF NOT EXISTS`, `DELETE`, and `INSERT` statements for `E$_` tables should be added according to Section 9.2 of the system prompt.
4.  **`snp_check_tab`:** Similarly, `snp_check_tab` operations were not present in the provided tasks. If this audit table is used for validation or logging, relevant cells should be added.
5.  **`SCEN_TASK_NO` Markers:** Tasks {1}, {4}, {5}, {7}, {10}, {20}, {70}, {140}, {170}, {190}, {200} were either parameter setup, skipped commands, or implicit commits/markers in the original ODI and have been handled as comments or absorbed into adjacent cells/Databricks conventions (e.g., implicit commit, widgets for parameters).
6.  **Widget Default Values:** Default values for widgets are provided (e.g., `1900-01-01` for timestamps, `-1` for IDs). These should be reviewed and adjusted based on the specific environment and initial load requirements.